In [ ]:
# Importing the required libraries
import pandas as pd
from google.colab import files


# 2020 WAEC DATA

## A. Loading the Dataset

In [ ]:
# Loading the DataFrame
raw_20 = pd.read_csv("/content/2020__WAEC_Data.csv", header=None)


In [ ]:
# Viewing the number of rows and columns in the raw DataFrame
raw_20.shape


## B. Defining a function to clean numeric columns

In [ ]:
def clean_numeric(df, num_cols):
  for col in num_cols:
    df[col] = (
        df[col]
        .astype("string")
        .str.replace(",", "", regex=False)
    )

    df[col] = pd.to_numeric(
        df[col], errors="coerce"
    )

  return df


## C. Defining a function to clean state names

In [ ]:
def clean_states(df):
  df["state"] = df["state"].astype("string").str.title()

  # Replacing Abuja with FCT
  df["state"] = df["state"].replace({
      "Abuja": "FCT"
  })

  # Removing Offshore
  df = df[
      ~df["state"].isin(["Offshore"])
  ]

  # Removing rows where state is missing
  df = df[df["state"].notna()]

  # Separating out the "Total" row so it isn't sorted alphabetically
  total_row = df[df["state"] == "Total"]
  df = df[df["state"] != "Total"]

  # Sorting the remaining states alphabetically
  df = df.sort_values("state")

  # Appending "Total" back as the last row
  df = pd.concat([df, total_row])

  # Resetting index
  df = df.reset_index(drop=True)

  return df


## D. Defining a function to label the missing "Total" row

As with 2019, several blocks in this sheet leave the grand-total row's
"state" cell blank instead of writing "Total", a side effect of the
original spreadsheet's merged header cells. Left alone, `clean_states`
would silently drop that row along with the genuinely blank filler rows,
since both have a missing state.

This function tells the two apart: the real total row still holds an actual,
comma-formatted number in another column, while filler rows are empty
everywhere. Wherever that pattern shows up, the row is labelled "Total" so
`clean_states` keeps it and sorts it to the bottom like every other block.

In [ ]:
def label_total_row(df, check_col):
  # Row is the grand-total row if its state is missing but it still holds
  # a real comma-formatted number (not a genuinely blank filler row)
  numeric_pattern = df[check_col].astype(str).str.match(r"^[\d,]+(\.\d+)?$")
  mask = df["state"].isna() & numeric_pattern

  df.loc[mask, "state"] = "Total"

  return df


## Creating the Private Students DataFrame

In this file, the simple state/male/female/total block for private
candidates sits at columns 14-17, rather than at the start of the sheet
as it did in 2021 and 2019 — the 2020 report orders its sections
differently, so the column positions below reflect this sheet's own
layout rather than being copied from the other years' notebooks.

In [ ]:
priv_stud_20 = raw_20.iloc[4:, [14, 15, 16, 17]].copy()


In [ ]:
priv_stud_20.columns = [
    "state",
    "private_cand_male",
    "private_cand_female",
    "private_cand_total"
]


In [ ]:
priv_stud_20 = label_total_row(priv_stud_20, "private_cand_total")


In [ ]:
priv_stud_20 = clean_states(priv_stud_20)


In [ ]:
priv_stud_20 = clean_numeric(
    priv_stud_20, [
      "private_cand_male",
      "private_cand_female",
      "private_cand_total"
    ]
)


In [ ]:
# Viewing the result
priv_stud_20


## PRIVATE ENGLISH AND PRIVATE MATHS

This file combines the private "total sat" + English-credit + Maths-credit
figures into one single block (columns 1-10), rather than splitting them
into two separate blocks the way 2019 and 2021 do. The English and Maths
DataFrames below are both built from this one block, each keeping only the
credit columns relevant to that subject.

In [ ]:
priv_total_eng_maths_20 = raw_20.iloc[4:, [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]].copy()


In [ ]:
priv_total_eng_maths_20.columns = [
    "state",
    "priv_cand_male",
    "priv_cand_female",
    "priv_cand_total",
    "priv_cred_5_eng_male",
    "priv_cred_5_eng_female",
    "priv_cred_5_eng_total",
    "priv_cred_5_maths_male",
    "priv_cred_5_maths_female",
    "priv_cred_5_maths_total"
]


In [ ]:
priv_total_eng_maths_20 = label_total_row(priv_total_eng_maths_20, "priv_cand_total")


In [ ]:
priv_total_eng_maths_20 = clean_states(priv_total_eng_maths_20)


In [ ]:
priv_total_eng_maths_20 = clean_numeric(
    priv_total_eng_maths_20, [
      "priv_cand_male",
      "priv_cand_female",
      "priv_cand_total",
      "priv_cred_5_eng_male",
      "priv_cred_5_eng_female",
      "priv_cred_5_eng_total",
      "priv_cred_5_maths_male",
      "priv_cred_5_maths_female",
      "priv_cred_5_maths_total"
    ]
)


In [ ]:
priv_total_eng_maths_20


## PRIVATE ENGLISH AND MATHS (COMBINED, WITH PERCENTAGE)

In [ ]:
priv_eng_maths_20 = raw_20.iloc[4:, [21, 22, 23, 24, 25, 26, 27, 28]].copy()


In [ ]:
priv_eng_maths_20.columns = [
    "state",
    "priv_cand_male",
    "priv_cand_female",
    "priv_cand_total",
    "priv_cred_5_eng_maths_male",
    "priv_cred_5_eng_maths_female",
    "priv_cred_5_eng_maths_total",
    "priv_cred_5_eng_maths_pct"
]


This block has "OFFSHOR" instead of "Offshore" for the offshore-candidates
row (another truncated label from the source spreadsheet). It's fixed here
before cleaning so `clean_states` can correctly recognise and drop it,
the same way it already recognises "Offshore".

In [ ]:
# Fixing the truncated "Offshore" label in this block
priv_eng_maths_20["state"] = priv_eng_maths_20["state"].replace({
    "OFFSHOR": "OFFSHORE"
})


In [ ]:
priv_eng_maths_20 = label_total_row(priv_eng_maths_20, "priv_cand_total")


In [ ]:
priv_eng_maths_20 = clean_states(priv_eng_maths_20)


In [ ]:
priv_eng_maths_20 = clean_numeric(
    priv_eng_maths_20, [
      "priv_cand_male",
      "priv_cand_female",
      "priv_cand_total",
      "priv_cred_5_eng_maths_male",
      "priv_cred_5_eng_maths_female",
      "priv_cred_5_eng_maths_total",
      "priv_cred_5_eng_maths_pct"
    ]
)


In [ ]:
priv_eng_maths_20


## PUBLIC ENGLISH AND MATHS (COMBINED, WITH PERCENTAGE)

In [ ]:
pub_eng_maths_20 = raw_20.iloc[4:, [32, 33, 34, 35, 36, 37, 38, 39]].copy()


In [ ]:
pub_eng_maths_20.columns = [
    "state",
    "pub_cand_male",
    "pub_cand_female",
    "pub_cand_total",
    "pub_cred_5_eng_maths_male",
    "pub_cred_5_eng_maths_female",
    "pub_cred_5_eng_maths_total",
    "pub_cred_5_eng_maths_pct"
]


In [ ]:
pub_eng_maths_20 = label_total_row(pub_eng_maths_20, "pub_cand_total")


In [ ]:
pub_eng_maths_20 = clean_states(pub_eng_maths_20)


In [ ]:
pub_eng_maths_20 = clean_numeric(
    pub_eng_maths_20, [
      "pub_cand_male",
      "pub_cand_female",
      "pub_cand_total",
      "pub_cred_5_eng_maths_male",
      "pub_cred_5_eng_maths_female",
      "pub_cred_5_eng_maths_total",
      "pub_cred_5_eng_maths_pct"
    ]
)


In [ ]:
pub_eng_maths_20


## PUBLIC SCHOOL CANDIDATES

In [ ]:
pub_cands_20 = raw_20.iloc[4:, [43, 44, 45, 46]].copy()


In [ ]:
pub_cands_20.columns = [
    "state",
    "pub_cands_male",
    "pub_cands_female",
    "pub_cands_total"
]


In [ ]:
pub_cands_20 = label_total_row(pub_cands_20, "pub_cands_total")


In [ ]:
pub_cands_20 = clean_states(pub_cands_20)


In [ ]:
pub_cands_20 = clean_numeric(
    pub_cands_20, [
        "pub_cands_male",
        "pub_cands_female",
        "pub_cands_total"
    ]
)


In [ ]:
pub_cands_20


## PUBLIC ENGLISH AND PUBLIC MATHS

In [ ]:
pub_total_eng_maths_20 = raw_20.iloc[4:, [50, 51, 52, 53, 54, 55, 56, 57, 58, 59]].copy()


In [ ]:
pub_total_eng_maths_20.columns = [
    "state",
    "pub_cands_male",
    "pub_cands_female",
    "pub_cands_total",
    "pub_creds_5_eng_male",
    "pub_creds_5_eng_female",
    "pub_creds_5_eng_total",
    "pub_creds_5_maths_male",
    "pub_creds_5_maths_female",
    "pub_creds_5_maths_total"
]


In [ ]:
pub_total_eng_maths_20 = label_total_row(pub_total_eng_maths_20, "pub_cands_total")


In [ ]:
pub_total_eng_maths_20 = clean_states(pub_total_eng_maths_20)


In [ ]:
pub_total_eng_maths_20 = clean_numeric(
    pub_total_eng_maths_20, [
      "pub_cands_male",
      "pub_cands_female",
      "pub_cands_total",
      "pub_creds_5_eng_male",
      "pub_creds_5_eng_female",
      "pub_creds_5_eng_total",
      "pub_creds_5_maths_male",
      "pub_creds_5_maths_female",
      "pub_creds_5_maths_total"
    ]
)


In [ ]:
pub_total_eng_maths_20


## COMBINING THE PRIVATE DATAFRAMES

In [ ]:
priv_waec_20 = (
    priv_stud_20
    .merge(
        priv_total_eng_maths_20[
            [
                "state",
                "priv_cred_5_eng_male",
                "priv_cred_5_eng_female",
                "priv_cred_5_eng_total"
            ]
        ],
        on="state",
        how="outer"
    )
    .merge(
        priv_total_eng_maths_20[
            [
                "state",
                "priv_cred_5_maths_male",
                "priv_cred_5_maths_female",
                "priv_cred_5_maths_total"
            ]
        ],
        on="state",
        how="outer"
    )
    .merge(
        priv_eng_maths_20[
            [
                "state",
                "priv_cred_5_eng_maths_male",
                "priv_cred_5_eng_maths_female",
                "priv_cred_5_eng_maths_total",
                "priv_cred_5_eng_maths_pct"
            ]
        ],
        on="state",
        how="outer"
    )
)


In [ ]:
priv_waec_20


In [ ]:
# Dropping the states with missing values
priv_waec_20 = priv_waec_20.dropna(axis=0, how="any")


In [ ]:
# Sorting 'Total' to the bottom
# Separating out the "Total" row so it isn't sorted alphabetically
total_row = priv_waec_20[priv_waec_20["state"] == "Total"]
priv_waec_20 = priv_waec_20[priv_waec_20["state"] != "Total"]

# Sorting the remaining states alphabetically
priv_waec_20 = priv_waec_20.sort_values("state")

# Appending "Total" back as the last row
priv_waec_20 = pd.concat([priv_waec_20, total_row])

# Resetting the index
priv_waec_20 = priv_waec_20.reset_index(drop=True)


In [ ]:
priv_waec_20


In [ ]:
file_name = "Cleaned 2020 Private WAEC Data.csv"

priv_waec_20.to_csv(file_name, index=False)
files.download(file_name)


## COMBINING PUBLIC

In [ ]:
pub_waec_20 = (
    pub_cands_20
    .merge(
        pub_total_eng_maths_20[
            [
                "state",
                "pub_creds_5_eng_male",
                "pub_creds_5_eng_female",
                "pub_creds_5_eng_total"
            ]
        ],
        on="state",
        how="outer"
    )
    .merge(
        pub_total_eng_maths_20[
            [
                "state",
                "pub_creds_5_maths_male",
                "pub_creds_5_maths_female",
                "pub_creds_5_maths_total"
            ]
        ],
        on="state",
        how="outer"
    )
    .merge(
        pub_eng_maths_20[
            [
                "state",
                "pub_cred_5_eng_maths_male",
                "pub_cred_5_eng_maths_female",
                "pub_cred_5_eng_maths_total",
                "pub_cred_5_eng_maths_pct"
            ]
        ],
        on="state",
        how="outer"
    )
)


In [ ]:
pub_waec_20


In [ ]:
pub_waec_20 = pub_waec_20.dropna(axis=0, how="any")


In [ ]:
total_row = pub_waec_20[pub_waec_20["state"] == "Total"]
pub_waec_20 = pub_waec_20[pub_waec_20["state"] != "Total"]

pub_waec_20 = pd.concat([pub_waec_20.sort_values("state"), total_row]).reset_index(drop=True)


In [ ]:
pub_waec_20


In [ ]:
file_name = "Cleaned 2020 Public WAEC Data.csv"

pub_waec_20.to_csv(file_name, index=False)
files.download(file_name)
